<a href="https://colab.research.google.com/github/ibrahimhanifceker/YZ50Codes/blob/main/YZ50Hafta9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [211]:
sentence = "ÇİLEKleri gördüm ağzım sulandı"
print(sentence)
print(f"length: {len(sentence)}")
tokens = list(sentence.encode("utf-8"))
print(tokens)
print(f"length: {len(tokens)}")
print(f"İ: {list("İ".encode("utf-8"))}")
print(f"ü: {list("ü".encode("utf-8"))}")
print(f"ğ: {list("ğ".encode("utf-8"))}")
print(f"Ç: {list("Ç".encode("utf-8"))}")
print(f"E: {list("E".encode("utf-8"))}")
# Kod noktalarini direkt token olarak kullanmak mantikli degil cunku context lengthi cok uzun oluyor sadece 256 cesit token olunca.

ÇİLEKleri gördüm ağzım sulandı
length: 30
[195, 135, 196, 176, 76, 69, 75, 108, 101, 114, 105, 32, 103, 195, 182, 114, 100, 195, 188, 109, 32, 97, 196, 159, 122, 196, 177, 109, 32, 115, 117, 108, 97, 110, 100, 196, 177]
length: 37
İ: [196, 176]
ü: [195, 188]
ğ: [196, 159]
Ç: [195, 135]
E: [69]


In [212]:
import os

if not os.path.exists("tr_dataset.txt"):
    !wget https://raw.githubusercontent.com/ibrahimhanifceker/YZ50Codes/refs/heads/main/hafta-8/tr_dataset.txt

In [213]:
with open('tr_dataset.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(text[:10])
tokens = list(text.encode("utf-8"))
print(len(tokens))

Sermet Bey
181788


In [214]:
def get_max_pair(token_list):
    pair_count = {}
    max_pair = (-1, -1)
    for token_pair in zip(token_list, token_list[1:]):
        new_count = pair_count.get(token_pair, 0) + 1
        if new_count > pair_count.get(max_pair, 0):
            max_pair = token_pair
        pair_count[token_pair] = new_count
    # print([(bytes(k).decode("utf-8", errors="ignore"), v) for v, k in sorted(((v, k) for k, v in pair_count.items()), reverse=True)])
    return max_pair

pair = get_max_pair(tokens)
print(pair)
print(bytes(pair).decode("utf-8"))

(196, 177)
ı


In [215]:
def merge_tokens(token_list, target_pair, new_id):
    ind = 0
    new_tokens = []
    while ind < len(token_list):
        if ind < len(token_list) - 1 and target_pair == tuple(token_list[ind:ind+2]):
            new_tokens.append(new_id)
            ind += 2
        else:
            new_tokens.append(token_list[ind])
            ind += 1
    return new_tokens

In [216]:
vocab_size = 300
num_merges = vocab_size - 256

tokens_ = list(tokens)
merges = {}

for id in range(num_merges):
    max_pair = get_max_pair(tokens_)
    new_id = id + 256
    tokens_ = merge_tokens(tokens_, max_pair, new_id)
    merges[max_pair] = new_id
    if id < 20:
        print(f"byte pair: {max_pair}, \t new token id: {new_id} \t char pair: \"{bytes(max_pair).decode()}\" ")

print(f"Compression ratio: {len(tokens) / len(tokens_): .2f}")

byte pair: (196, 177), 	 new token id: 256 	 char pair: "ı" 
byte pair: (195, 188), 	 new token id: 257 	 char pair: "ü" 
byte pair: (110, 32), 	 new token id: 258 	 char pair: "n " 
byte pair: (101, 32), 	 new token id: 259 	 char pair: "e " 
byte pair: (97, 114), 	 new token id: 260 	 char pair: "ar" 
byte pair: (197, 159), 	 new token id: 261 	 char pair: "ş" 
byte pair: (97, 32), 	 new token id: 262 	 char pair: "a " 
byte pair: (46, 32), 	 new token id: 263 	 char pair: ". " 
byte pair: (101, 114), 	 new token id: 264 	 char pair: "er" 
byte pair: (105, 32), 	 new token id: 265 	 char pair: "i " 
byte pair: (44, 32), 	 new token id: 266 	 char pair: ", " 
byte pair: (105, 110), 	 new token id: 267 	 char pair: "in" 
byte pair: (105, 114), 	 new token id: 268 	 char pair: "ir" 
byte pair: (195, 167), 	 new token id: 269 	 char pair: "ç" 
byte pair: (97, 110), 	 new token id: 270 	 char pair: "an" 
byte pair: (196, 159), 	 new token id: 271 	 char pair: "ğ" 
byte pair: (195, 182), 	

In [217]:
vocab = {id: bytes([id]) for id in range(256)}
for (p0, p1), new_id in merges.items():
    vocab[new_id] = vocab[p0] + vocab[p1]

def decode(token_list):
    return b"".join([vocab[token] for token in token_list]).decode("utf-8", errors="replace")

def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        max_pair = min({pair for pair in zip(tokens, tokens[1:])}, key=lambda p: merges.get(p, float("inf")))
        if max_pair not in merges:
            break
        tokens = merge_tokens(tokens, max_pair, merges[max_pair])
    return tokens

sentence = "ÇİLEKleri gördüm 🍓 ağzım sulandı 😋"
decode(encode(sentence))

'ÇİLEKleri gördüm 🍓 ağzım sulandı 😋'

In [218]:
import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

In [219]:
groups = re.findall(gpt2pat, text)
tokens = [list(group.encode("utf-8")) for group in groups]

In [220]:
def get_max_group_pair(token_groups):
    pair_count = {}
    max_pair = (-1, -1)
    for group in token_groups:
        for token_pair in zip(group, group[1:]):
            new_count = pair_count.get(token_pair, 0) + 1
            if new_count > pair_count.get(max_pair, 0):
                max_pair = token_pair
            pair_count[token_pair] = new_count
    return max_pair

get_max_group_pair(tokens)

(196, 177)

In [221]:
def merge_group_tokens(token_group, target_pair, new_id):
    new_tokens = []
    for group in token_group:
        ind = 0
        new_group_tokens = []
        while ind < len(group):
            if ind < len(group) - 1 and target_pair == tuple(group[ind:ind+2]):
                new_group_tokens.append(new_id)
                ind += 2
            else:
                new_group_tokens.append(group[ind])
                ind += 1
        new_tokens.append(new_group_tokens)
    return new_tokens

In [222]:
vocab_size = 300
num_merges = vocab_size - 256

tokens_ = list(tokens)
merges = {}

for id in range(num_merges):
    max_pair = get_max_group_pair(tokens_)
    new_id = id + 256
    tokens_ = merge_group_tokens(tokens_, max_pair, new_id)
    merges[max_pair] = new_id
    if id < 20:
        print(f"byte pair: {max_pair}, \t new token id: {new_id}")

print(f"Compression ratio: {sum(len(t) for t in tokens) / sum(len(t) for t in tokens_): .2f}")

byte pair: (196, 177), 	 new token id: 256
byte pair: (195, 188), 	 new token id: 257
byte pair: (32, 98), 	 new token id: 258
byte pair: (97, 114), 	 new token id: 259
byte pair: (197, 159), 	 new token id: 260
byte pair: (97, 110), 	 new token id: 261
byte pair: (105, 110), 	 new token id: 262
byte pair: (101, 110), 	 new token id: 263
byte pair: (101, 114), 	 new token id: 264
byte pair: (32, 107), 	 new token id: 265
byte pair: (32, 100), 	 new token id: 266
byte pair: (105, 114), 	 new token id: 267
byte pair: (195, 167), 	 new token id: 268
byte pair: (32, 103), 	 new token id: 269
byte pair: (196, 159), 	 new token id: 270
byte pair: (256, 110), 	 new token id: 271
byte pair: (195, 182), 	 new token id: 272
byte pair: (97, 107), 	 new token id: 273
byte pair: (32, 121), 	 new token id: 274
byte pair: (32, 115), 	 new token id: 275
Compression ratio:  1.41


In [223]:
print(re.findall(gpt2pat, "Ankara'dan İstanbul'a giderken Ahmet'le karşılaştım."))

['Ankara', "'d", 'an', ' İstanbul', "'", 'a', ' giderken', ' Ahmet', "'", 'le', ' karşılaştım', '.']


In [224]:
# desen turkce icin dogru calismiyor cunku ayrilan ekler farkli bicimler alabiliyor mesela 'dan' eki 'd' ve 'an' diye ayrildi.

In [225]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

In [226]:
text_en = "Unicode has largely supplanted the previous environment of myriad incompatible character sets used within different locales and on different computer architectures. The entire repertoire of these sets, plus many additional characters, were merged into the single Unicode set. Unicode is used to encode the vast majority of text on the Internet, including most web pages, and relevant Unicode support has become a common consideration in contemporary software development. Unicode is ultimately capable of encoding more than 1.1 million characters."
tokens_en = enc.encode(text_en)
text_tr = "Standardın içinde karakterler ve karakterlere atanmış sayı değerlerinin tablolaştırılmış hali, bu sayılarının kodlanmasıyla ilgili kurum tarafından önerilen standart kodlama sistemleri ve bunların yanı sıra eşdeğer karakterler, karakterin bileşenlerine ayrılış bilgileri, sıralama kuralı, büyük-küçük harf bilgisi, yazılış yönü bilgisi gibi karakterin ekranda doğru gösterilebilmesi için yazılımların ihtiyaç duyduğu ek bilgiler bulunmaktadır.[1] Haziran 2015 tarihi itibarıyla standardın en son sürümü olan Unicode 8.0 ile birlikte 7.716 yeni karakter eklemesi yapılmıştır[2]"
tokens_tr = enc.encode(text_tr)
print(len(text_en), len(text_tr))
print(len(tokens_en), len(tokens_tr))

print(len(encode(text_tr)))

547 576
91 204
433
